# Daily data diagnostic and demo export

This notebook is the preparation layer for the supervisor-facing EDA notebook.

It is the only notebook in this workflow that reads SQLite and runs SQL. It:

1. selects a frozen daily-candle run;
2. extracts a small market manifest and the native daily candle rows;
3. checks basic integrity and coverage;
4. writes stable CSV artifacts under data/demo/;
5. writes a small JSON provenance summary.

The downstream EDA notebook reads only these exported files.

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import json
import sqlite3

import numpy as np
import pandas as pd
from IPython.display import Markdown, display

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_colwidth", 120)

PROJECT_ROOT = next(
    (
        path
        for path in [Path.cwd(), *Path.cwd().parents]
        if (path / "db/kalshi_daily_probability_dataset.sqlite").exists()
    ),
    Path.cwd(),
)
DB_PATH = PROJECT_ROOT / "db/kalshi_daily_probability_dataset.sqlite"
DATA_DIR = PROJECT_ROOT / "data" / "demo"
DATA_DIR.mkdir(parents=True, exist_ok=True)

SELECTION_ID = "20260817T124811Z-3b15c8a0-637b555156e3"
SELECTION_GROUP = "non_sport_crypto"
CANDLE_RUN_ID = None

MANIFEST_PATH = DATA_DIR / "non_sport_crypto_market_manifest.csv"
CANDLES_PATH = DATA_DIR / "non_sport_crypto_daily_candles.csv.gz"
SUMMARY_PATH = DATA_DIR / "export_summary.json"

print("PROJECT_ROOT:", PROJECT_ROOT)
print("DB_PATH:", DB_PATH)
print("DATA_DIR:", DATA_DIR)
print("SELECTION_ID:", SELECTION_ID)
print("SELECTION_GROUP:", SELECTION_GROUP)

In [ ]:
def read_sql(query, params=()):
    read_uri = f"file:{DB_PATH.resolve()}?mode=ro"
    with sqlite3.connect(read_uri, uri=True, timeout=60) as conn:
        return pd.read_sql_query(query, conn, params=params)

def parse_json(value):
    try:
        return json.loads(value) if value else {}
    except (TypeError, json.JSONDecodeError):
        return {}

runs = read_sql(
    """
    SELECT run_id, started_at_utc, finished_at_utc, status, source_mode,
           config_json, stats_json, error_text
    FROM metadata_runs
    WHERE json_extract(config_json, '$.stage') = 'daily_candles'
      AND json_extract(config_json, '$.selection_id') = ?
      AND json_extract(config_json, '$.selection_group') = ?
    ORDER BY started_at_utc DESC
    """,
    (SELECTION_ID, SELECTION_GROUP),
)

if CANDLE_RUN_ID is None:
    if runs.empty:
        raise RuntimeError("No daily-candle run found for the selected group.")
    selected_run = runs.iloc[0]
else:
    selected = runs.loc[runs["run_id"].eq(CANDLE_RUN_ID)]
    if selected.empty:
        raise RuntimeError(f"CANDLE_RUN_ID not found: {CANDLE_RUN_ID}")
    selected_run = selected.iloc[0]

CANDLE_RUN_ID = selected_run["run_id"]
RUN_STATUS = selected_run["status"]
RUN_CONFIG = parse_json(selected_run["config_json"])
RUN_STATS = parse_json(selected_run["stats_json"])

display(runs[[
    "run_id", "status", "source_mode", "started_at_utc",
    "finished_at_utc", "error_text"
]].head(10))

print("Selected CANDLE_RUN_ID:", CANDLE_RUN_ID)
print("Selected run status:", RUN_STATUS)

## 1. Extract the market manifest

The manifest has one row per market selected for the daily-candle request. It retains the pre-download filter decisions and the market lifecycle metadata needed by the EDA notebook.

In [ ]:
manifest = read_sql(
    """
    SELECT
        mh.market_id,
        mh.market_ticker,
        mh.event_ticker,
        mh.series_ticker,
        mh.selection_id,
        mh.selection_group,
        mh.filter_mode,
        mh.passes_volume_filter,
        mh.passes_lifetime_filter,
        mh.passes_both_filters,
        mh.volume_fp,
        mh.lifetime_days,
        mh.history_start_ts,
        mh.history_end_ts,
        mh.expected_daily_rows,
        mh.received_daily_rows,
        mh.first_observation_ts,
        mh.last_observation_ts,
        mh.status,
        mh.error_text,
        m.title,
        m.status AS market_status,
        m.result,
        m.open_time,
        m.close_time
    FROM market_history_manifest mh
    LEFT JOIN raw_markets m ON m.market_id = mh.market_id
    WHERE mh.run_id = ? AND mh.selection_id = ? AND mh.selection_group = ?
    ORDER BY mh.market_ticker
    """,
    (CANDLE_RUN_ID, SELECTION_ID, SELECTION_GROUP),
)

if manifest.empty:
    raise RuntimeError("The selected candle run has an empty market manifest.")

for column in [
    "volume_fp", "lifetime_days", "expected_daily_rows",
    "received_daily_rows", "history_start_ts", "history_end_ts",
    "first_observation_ts", "last_observation_ts",
]:
    manifest[column] = pd.to_numeric(manifest[column], errors="coerce")

for column in ["open_time", "close_time"]:
    manifest[column] = pd.to_datetime(manifest[column], utc=True, errors="coerce")

manifest["candle_run_id"] = CANDLE_RUN_ID
manifest["actual_candles_exported"] = 0

display(manifest.head(10))
print("Manifest rows:", len(manifest))

## 2. Extract the native daily candle rows

The time-series export is intentionally small and presentation-oriented. It keeps the main source fields required for daily probability inspection, while the full raw payload remains recoverable in SQLite.

In [ ]:
candles = read_sql(
    """
    SELECT
        c.market_id,
        c.market_ticker,
        c.series_ticker,
        c.end_period_ts,
        date(c.end_period_ts, 'unixepoch') AS date_utc,
        c.period_interval,
        c.source_mode,
        c.price_open,
        c.price_low,
        c.price_high,
        c.price_close,
        c.price_mean,
        c.price_previous,
        c.price_min,
        c.price_max,
        c.yes_bid_close,
        c.yes_ask_close,
        c.volume,
        c.open_interest,
        c.request_start_ts,
        c.request_end_ts,
        c.retrieved_at_utc,
        c.run_id
    FROM raw_daily_candles c
    JOIN market_history_manifest mh
      ON mh.market_ticker = c.market_ticker
     AND mh.run_id = ?
     AND mh.selection_id = ?
     AND mh.selection_group = ?
    WHERE c.source_mode = 'live' AND c.run_id = ?
    ORDER BY c.market_ticker, c.end_period_ts
    """,
    (CANDLE_RUN_ID, SELECTION_ID, SELECTION_GROUP, CANDLE_RUN_ID),
)

for column in [
    "end_period_ts", "period_interval", "price_open", "price_low",
    "price_high", "price_close", "price_mean", "price_previous",
    "price_min", "price_max", "yes_bid_close", "yes_ask_close",
    "volume", "open_interest", "request_start_ts", "request_end_ts",
]:
    candles[column] = pd.to_numeric(candles[column], errors="coerce")

print("Candle rows:", len(candles))
print("Markets with candles:", candles["market_ticker"].nunique())
display(candles.head(10))

## 3. Quality checks and summary

The export is allowed to be partial because live downloads may have API errors. Partial status is preserved explicitly in the summary instead of being hidden.

In [ ]:
actual_counts = (
    candles.groupby("market_ticker")
    .size()
    .rename("actual_candles_exported")
    .reset_index()
)

manifest = manifest.drop(columns=["actual_candles_exported"]).merge(
    actual_counts,
    on="market_ticker",
    how="left",
)
manifest["actual_candles_exported"] = manifest["actual_candles_exported"].fillna(0).astype(int)

quality = {
    "duplicate_market_day_rows": int(
        candles.duplicated(["market_ticker", "end_period_ts"]).sum()
    ),
    "missing_market_ticker_rows": int(candles["market_ticker"].isna().sum()),
    "non_daily_interval_rows": int(candles["period_interval"].ne(1440).sum()),
    "close_below_zero_rows": int(candles["price_close"].lt(0).sum()),
    "close_above_one_rows": int(candles["price_close"].gt(1).sum()),
    "negative_volume_rows": int(candles["volume"].lt(0).sum()),
    "manifest_markets": int(manifest["market_ticker"].nunique()),
    "exported_markets": int(candles["market_ticker"].nunique()),
    "manifest_rows_with_zero_exported_candles": int(
        manifest["actual_candles_exported"].eq(0).sum()
    ),
}

if len(candles):
    first_date = str(candles["date_utc"].min())
    last_date = str(candles["date_utc"].max())
else:
    first_date = None
    last_date = None

filter_counts = (
    manifest.groupby(
        ["passes_volume_filter", "passes_lifetime_filter", "passes_both_filters"],
        dropna=False,
    )
    .size()
    .reset_index(name="markets")
)

summary = {
    "export_created_at_utc": datetime.now(timezone.utc).isoformat(),
    "selection_id": SELECTION_ID,
    "selection_group": SELECTION_GROUP,
    "candle_run_id": CANDLE_RUN_ID,
    "run_status": RUN_STATUS,
    "run_started_at_utc": selected_run["started_at_utc"],
    "run_finished_at_utc": selected_run["finished_at_utc"],
    "manifest_rows": int(len(manifest)),
    "candle_rows": int(len(candles)),
    "markets_with_candles": int(candles["market_ticker"].nunique()),
    "first_candle_date_utc": first_date,
    "last_candle_date_utc": last_date,
    "filter_counts": filter_counts.to_dict(orient="records"),
    "quality": quality,
    "run_stats": RUN_STATS,
}

display(pd.DataFrame([
    {"metric": key, "value": value}
    for key, value in summary.items()
    if key not in {"filter_counts", "quality", "run_stats"}
]))
display(pd.DataFrame([quality]))

## 4. Write the demo artifacts

The market manifest is a regular CSV. The time series is written as gzip-compressed CSV so the supervisor-facing notebook remains dependency-light and can still use pandas only.

In [ ]:
manifest_output = manifest.copy()
for column in ["open_time", "close_time"]:
    manifest_output[column] = manifest_output[column].dt.strftime("%Y-%m-%dT%H:%M:%SZ")

manifest_output.to_csv(MANIFEST_PATH, index=False)
candles.to_csv(CANDLES_PATH, index=False, compression="gzip")

with SUMMARY_PATH.open("w", encoding="utf-8") as handle:
    json.dump(summary, handle, indent=2, default=str)

print("Wrote:", MANIFEST_PATH)
print("Wrote:", CANDLES_PATH)
print("Wrote:", SUMMARY_PATH)
print("Manifest size (MB):", round(MANIFEST_PATH.stat().st_size / 1024**2, 2))
print("Candles size (MB):", round(CANDLES_PATH.stat().st_size / 1024**2, 2))

## Handoff to daily_data_eda.ipynb

The EDA notebook should read only:

- non_sport_crypto_market_manifest.csv;
- non_sport_crypto_daily_candles.csv.gz;
- export_summary.json.

If this diagnostic is rerun with another candle run, the same demo filenames are refreshed and the summary records the new run ID and status.